# CS-781 Phase 2 — Experiment 2 (v3 BEST): Model 1 + LoRA

**Main objective (unchanged):** adapt the Phase-1 **Model 1 (Prefix Encoder / Hybrid LM)** to Phase-2 Legal NER using **LoRA**, while keeping the original Phase-1 backbone weights frozen.

This version keeps the stronger v2 design and adds reliability/reproducibility checks around checkpoint selection, LoRA initialization, AMP scheduling, held-out evaluation, final refit, and submission generation.

**Experiment identity:** Model 1 + LoRA on attention (`qkv`, `out_proj`) and FFN (`ff.0`, `ff.2`) + BILOU NER head.

- Phase-1 base weights: **frozen**
- Trainable: **LoRA A/B matrices + NER classifier only**
- Native Model-1 full bidirectional attention is used for NER
- Fold1 = training, Fold2 = held-out selection, then fresh LoRA refit on Fold1+Fold2
- `PRESPLIT_PUNCT=True` is retained because it gave near-perfect span representability in Experiment 1
- Strict Macro-F1 is the model-selection metric


In [1]:
import os, sys, json, math, time, random, re, zipfile, subprocess, gc, hashlib
from pathlib import Path
from collections import Counter, defaultdict

try:
    import huggingface_hub
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download

SEED=781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
print("PyTorch:",torch.__version__)
print("Device:",DEVICE)
if DEVICE=="cuda": print("GPU:",torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [2]:
ROOT=Path("/kaggle/working/phase2_exp2_model1_lora")
ROOT.mkdir(parents=True,exist_ok=True)
HF_REPO="Exploration-Lab/CS-781-Capstone"

MODEL1_BEST_PATH=None
TOKENIZER_PATH=None

# ---------- NER / windowing ----------
NER_SEQ_LEN=768
WINDOW_OVERLAP=192
WINDOW_STEP=NER_SEQ_LEN-WINDOW_OVERLAP
TRAIN_BATCH_SIZE=2
EVAL_BATCH_SIZE=4
GRAD_ACCUM_STEPS=8
EPOCHS=12

# ---------- LoRA ----------
LORA_R=32
LORA_ALPHA=64
LORA_DROPOUT=0.10
LORA_TARGET_QKV=True
LORA_TARGET_OUT=True
LORA_TARGET_FFN=True    # stronger LoRA: adapt ff.0 and ff.2 as well; still one LoRA experiment
LORA_LR=5e-4
HEAD_LR=1e-3
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10
MAX_GRAD_NORM=1.0
HEAD_DROPOUT=0.10

USE_CLASS_WEIGHTS=True
AMP=True
USE_VITERBI=True
PRESPLIT_PUNCT=True
TOKENIZER_CHECK_DOCS=12
USE_PER_LABEL_THRESHOLDS=True
THRESHOLD_SMOOTH=True       # smooth the per-label F1-vs-threshold curve before taking the argmax
THRESHOLD_GRID=[round(x,2) for x in np.arange(0.00,0.81,0.05)]

# After Fold2 selects BEST_EPOCH + thresholds, automatically refit a fresh LoRA adapter on all 70 labelled docs.
DO_FINAL_REFIT_ON_ALL=True

EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

BEST_NER_PATH=ROOT/"best_lora_ner.pt"
VAL_PRED_PATH=ROOT/"val_prediction.json"
FINAL_REFIT_PATH=ROOT/"final_refit_lora_ner.pt"
TEST_PRED_PATH=ROOT/"prediction.json"
SUBMISSION_ZIP=ROOT/"submission.zip"
EVAL_SCRIPT_PATH=ROOT/"evaluate_phase2.py"

print("ROOT:",ROOT)
print("NER seq len:",NER_SEQ_LEN,"overlap:",WINDOW_OVERLAP,"step:",WINDOW_STEP)
print(f"LoRA: r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT} targets=qkv+out+ffn")


ROOT: /kaggle/working/phase2_exp2_model1_lora
NER seq len: 768 overlap: 192 step: 576
LoRA: r=32 alpha=64 dropout=0.1 targets=qkv+out+ffn


In [3]:
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def find_files():
    ckpts,toks=[],[]
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.exists(base): continue
        for root,dirs,files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn); low=fn.lower()
                if fn.endswith(".pt") and "enc_full_best" in low: ckpts.append(p)
                if low=="tokenizer.json" or ("tokenizer" in low and low.endswith(".json")): toks.append(p)
    return sorted(ckpts),sorted(toks)

ckpts,toks=find_files()
print("Checkpoint candidates:")
for x in ckpts: print(" ",x)
print("\nTokenizer candidates:")
for x in toks: print(" ",x,"sha=",sha256_file(x))

if MODEL1_BEST_PATH is None:
    # Prefer the exact known Model-1 best checkpoint if several copies/checkpoints are attached.
    exact=[x for x in ckpts if "enc_full_best (9).pt" in os.path.basename(x).lower()]
    preferred=[x for x in ckpts if "phase1-complete-model" in x.lower()]
    MODEL1_BEST_PATH=(exact[-1] if exact else (preferred[-1] if preferred else (ckpts[-1] if ckpts else None)))

if TOKENIZER_PATH is None:
    matching=[x for x in toks if sha256_file(x)==EXPECTED_TOKENIZER_SHA256]
    if matching: TOKENIZER_PATH=matching[-1]

assert MODEL1_BEST_PATH and os.path.exists(MODEL1_BEST_PATH), (
    "Model-1 enc_full_best checkpoint not found. Attach enc_full_best (9).pt or set MODEL1_BEST_PATH manually."
)
assert TOKENIZER_PATH and os.path.exists(TOKENIZER_PATH), (
    "Verified tokenizer.json not found. Attach the tokenizer whose SHA-256 is the expected value above."
)

print("\nUsing Model-1 checkpoint:",MODEL1_BEST_PATH)
print("Model-1 checkpoint SHA-256:",sha256_file(MODEL1_BEST_PATH))
print("Using tokenizer:",TOKENIZER_PATH)
tok_sha=sha256_file(TOKENIZER_PATH)
print("Tokenizer SHA-256:",tok_sha)
assert tok_sha==EXPECTED_TOKENIZER_SHA256, f"Tokenizer mismatch: {tok_sha}"
print("✅ tokenizer.json is byte-for-byte identical to the verified Phase-1 tokenizer.")


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt

Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json sha= 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d

Using Model-1 checkpoint: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
Model-1 checkpoint SHA-256: 017c57f393a55c96a9a24d0f98fc62775f3f4d0459b896eefe7a30437accacfc
Using tokenizer: /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
Tokenizer SHA-256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d
✅ tokenizer.json is byte-for-byte identical to the verified Phase-1 tokenizer.


In [4]:
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]

class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]; self.itos=list(BPE_SPECIALS)
        self.stoi={t:i for i,t in enumerate(self.itos)}; self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word); ranks=self._ranks
        while len(symbols)>1:
            best=None; best_rank=None
            for j in range(len(symbols)-1):
                r=ranks.get((symbols[j],symbols[j+1]))
                if r is not None and (best_rank is None or r<best_rank):
                    best=(symbols[j],symbols[j+1]); best_rank=r
            if best is None: break
            left,right=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==left and symbols[j+1]==right:
                    merged.append(left+right); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",text):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok in BPE_SPECIALS: continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        obj=json.load(open(path,encoding="utf-8")); tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]; tok.itos=list(obj["itos"])
        tok.stoi={t:i for i,t in enumerate(tok.itos)}; tok.vocab_size=len(tok.itos)
        tok.symbol_bytes={}
        for b in range(256): tok.symbol_bytes[f"{b:02x}"]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_PATH)
print("Tokenizer vocab:",len(tokenizer))

UNIT_RE=re.compile(r"\w+|[^\w\s]|\s+" if PRESPLIT_PUNCT else r"\S+|\s+")
print("Token unit regex:",UNIT_RE.pattern)

def encode_with_char_offsets(text):
    ids=[]; offsets=[]; exact=[]; unk=tokenizer.unk_token_id
    for m in UNIT_RE.finditer(text):
        unit=m.group(0); base=m.start(); symbols=tokenizer.encode_word(unit)
        byte_to_char={0:0}; bpos=0
        for ci,ch in enumerate(unit):
            bpos+=len(ch.encode("utf-8")); byte_to_char[bpos]=ci+1
        boundaries=sorted(byte_to_char)
        sb=0
        for sym in symbols:
            raw=tokenizer.symbol_bytes.get(sym)
            if raw is None:
                ids.append(unk); offsets.append((m.start(),m.end())); exact.append(False); continue
            eb=sb+len(raw); ok=(sb in byte_to_char and eb in byte_to_char)
            if ok:
                cs=base+byte_to_char[sb]; ce=base+byte_to_char[eb]
            else:
                lb=max(x for x in boundaries if x<=sb); rb=min(x for x in boundaries if x>=eb)
                cs=base+byte_to_char[lb]; ce=base+byte_to_char[rb]
            ids.append(tokenizer.stoi.get(sym,unk)); offsets.append((cs,ce)); exact.append(ok); sb=eb
    return ids,offsets,exact

probe="IN THE SUPREME COURT OF INDIA\\nSection 14: ₹100 — legal test."
pids,poff,pexact=encode_with_char_offsets(probe)
assert tokenizer.decode(pids)==probe
print("Round-trip exact: True")
print("Probe exact-boundary tokens:",sum(pexact),"/",len(pexact))


Tokenizer vocab: 24576
Token unit regex: \w+|[^\w\s]|\s+
Round-trip exact: True
Probe exact-boundary tokens: 27 / 27


In [5]:
PHASE2_FILES={
    "fold1_text":"Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":"Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":"Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":"Phase_2_LNER/Train/fold2-gold.json",
    "labels":"Phase_2_LNER/Train/labels.txt",
    "test":"Phase_2_LNER/Test/fold3-text.json",
}
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        print("✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).")
except Exception:
    print("ℹ️ HF_TOKEN secret not available; trying public Hugging Face access.")

def get_hf_file(filename):
    return hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=filename,
        token=HF_TOKEN if HF_TOKEN else None,
    )
phase2_paths={k:get_hf_file(v) for k,v in PHASE2_FILES.items()}
for k,v in phase2_paths.items(): print(f"{k:12s} -> {v}")

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

with open(phase2_paths["labels"],encoding="utf-8") as f:
    LABELS=[x.strip() for x in f if x.strip()]
assert len(LABELS)==12 and len(set(LABELS))==12,LABELS
print("Official labels:",LABELS)


✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

fold1_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
fold1_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
fold2_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
fold2_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
labels       -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
test         -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb

In [6]:
def normalize_text_json(obj):
    if isinstance(obj,dict):
        if all(isinstance(v,str) for v in obj.values()): return {str(k):v for k,v in obj.items()}
        if "data" in obj: return normalize_text_json(obj["data"])
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            txt=r.get("text",r.get("document"))
            if did is None or not isinstance(txt,str): raise ValueError(f"Cannot parse text record: {r}")
            out[str(did)]=txt
        return out
    raise TypeError(f"Unsupported text JSON type: {type(obj)}")

def normalize_one_entity(e):
    if isinstance(e,list):
        if len(e) not in (3,4): raise ValueError(f"Bad entity: {e}")
        a,b,l=e[:3]; et=e[3] if len(e)==4 else None
        return int(a),int(b),str(l),et
    if isinstance(e,dict):
        a=e.get("start",e.get("start_char",e.get("start_character")))
        b=e.get("end",e.get("end_char",e.get("end_character")))
        l=e.get("label",e.get("type",e.get("entity_type")))
        et=e.get("text",e.get("entity_text"))
        if a is None or b is None or l is None: raise ValueError(f"Cannot parse entity: {e}")
        return int(a),int(b),str(l),et
    raise TypeError(type(e))

def normalize_gold_json(obj):
    if isinstance(obj,dict) and "data" in obj: return normalize_gold_json(obj["data"])
    if isinstance(obj,dict):
        out={}
        for did,ents in obj.items():
            if isinstance(ents,dict): ents=ents.get("entities",ents.get("annotations",ents.get("spans")))
            if ents is None: ents=[]
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            ents=r.get("entities",r.get("annotations",r.get("spans")))
            if did is None or ents is None: raise ValueError(f"Cannot parse gold record: {r}")
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    raise TypeError(type(obj))

fold1_text=normalize_text_json(read_json(phase2_paths["fold1_text"]))
fold2_text=normalize_text_json(read_json(phase2_paths["fold2_text"]))
test_text=normalize_text_json(read_json(phase2_paths["test"]))
fold1_gold=normalize_gold_json(read_json(phase2_paths["fold1_gold"]))
fold2_gold=normalize_gold_json(read_json(phase2_paths["fold2_gold"]))
assert set(fold1_text)==set(fold1_gold)
assert set(fold2_text)==set(fold2_gold)

allowed=set(LABELS)
def validate_gold(text_map,gold_map,name):
    n_ent=0; bad_text=0; overlaps=0; counts=Counter(); lengths=[]
    for did,text in text_map.items():
        prev=[]
        for a,b,l,et in gold_map[did]:
            assert 0<=a<b<=len(text),(did,a,b,len(text))
            assert l in allowed,(did,l)
            if et is not None and text[a:b]!=et: bad_text+=1
            for aa,bb,ll,_ in prev:
                if max(a,aa)<min(b,bb): overlaps+=1
            prev.append((a,b,l,et)); n_ent+=1; counts[l]+=1; lengths.append(b-a)
    print(f"{name}: docs={len(text_map):,} entities={n_ent:,} entity_text_mismatch={bad_text} overlapping_pairs={overlaps}")
    print(" label counts:",dict(counts))
    if lengths:
        print(" entity char length p50/p90/max:",int(np.percentile(lengths,50)),int(np.percentile(lengths,90)),max(lengths))

validate_gold(fold1_text,fold1_gold,"Fold1")
validate_gold(fold2_text,fold2_gold,"Fold2")
print("Test docs:",len(test_text))


Fold1: docs=35 entities=4,120 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'RESP': 191, 'DATE': 813, 'COURT': 455, 'WIT': 67, 'APP': 207, 'STAT': 525, 'A.COUNSEL': 70, 'PREC': 624, 'JUDGE': 123, 'R.COUNSEL': 87, 'CASENO': 423, 'AUTH': 535}
 entity char length p50/p90/max: 13 34 195
Fold2: docs=35 entities=4,273 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'JUDGE': 122, 'CASENO': 337, 'APP': 200, 'RESP': 190, 'A.COUNSEL': 125, 'R.COUNSEL': 97, 'STAT': 735, 'DATE': 762, 'AUTH': 468, 'PREC': 559, 'COURT': 558, 'WIT': 120}
 entity char length p50/p90/max: 13 32 233
Test docs: 35


In [7]:
TAG_NAMES=["O"]
for lab in LABELS: TAG_NAMES += [f"B-{lab}",f"I-{lab}",f"L-{lab}",f"U-{lab}"]
TAG2ID={t:i for i,t in enumerate(TAG_NAMES)}
ID2TAG={i:t for t,i in TAG2ID.items()}
O_ID=TAG2ID["O"]; IGNORE_INDEX=-100; N_TAGS=len(TAG_NAMES)

def tag_document(text,entities):
    ids,offsets,exact=encode_with_char_offsets(text)
    tags=[O_ID]*len(ids); group=[-1]*len(ids)
    start_to_idx={}; end_to_idx={}
    for i,((a,b),ex) in enumerate(zip(offsets,exact)):
        if ex:
            start_to_idx.setdefault(a,i); end_to_idx[b]=i
    stats=Counter(); ranges=[]
    for gi,(a,b,lab,et) in enumerate(sorted(entities,key=lambda z:(z[0],z[1],z[2]))):
        if a not in start_to_idx or b not in end_to_idx:
            stats["unaligned"]+=1; continue
        i=start_to_idx[a]; j=end_to_idx[b]
        if j<i or offsets[i][0]!=a or offsets[j][1]!=b:
            stats["unaligned"]+=1; continue
        if not all(offsets[k][1]==offsets[k+1][0] for k in range(i,j)):
            stats["unaligned"]+=1; continue
        if any(group[k]!=-1 for k in range(i,j+1)):
            stats["overlap_conflict"]+=1; continue
        if i==j:
            tags[i]=TAG2ID[f"U-{lab}"]
        else:
            tags[i]=TAG2ID[f"B-{lab}"]
            for k in range(i+1,j): tags[k]=TAG2ID[f"I-{lab}"]
            tags[j]=TAG2ID[f"L-{lab}"]
        for k in range(i,j+1): group[k]=gi
        ranges.append((i,j,gi)); stats["aligned"]+=1
    return ids,offsets,tags,group,ranges,stats

def alignment_report(text_map,gold_map,name):
    total=Counter(); tok_lens=[]
    for did,text in text_map.items():
        ids,offsets,tags,group,ranges,st=tag_document(text,gold_map[did])
        total.update(st); tok_lens.append(len(ids))
    denom=total["aligned"]+total["unaligned"]+total["overlap_conflict"]
    print(f"{name} alignment:",dict(total))
    print("Representable gold spans:",f"{total['aligned']}/{denom} = {total['aligned']/max(1,denom):.4%}")
    print("Token lengths p50/p90/p95/max:",[int(np.percentile(tok_lens,q)) for q in (50,90,95)],max(tok_lens))

alignment_report(fold1_text,fold1_gold,"Fold1")
alignment_report(fold2_text,fold2_gold,"Fold2")


Fold1 alignment: {'aligned': 4093, 'unaligned': 27}
Representable gold spans: 4093/4120 = 99.3447%
Token lengths p50/p90/p95/max: [10181, 19167, 21017] 40485
Fold2 alignment: {'aligned': 4257, 'unaligned': 16}
Representable gold spans: 4257/4273 = 99.6256%
Token lengths p50/p90/p95/max: [10734, 16185, 24784] 44587


In [8]:
import bisect

class NERWindowDataset(Dataset):
    def __init__(self,text_map,gold_map=None,seq_len=NER_SEQ_LEN,step=WINDOW_STEP):
        self.text_map=text_map; self.gold_map=gold_map; self.seq_len=seq_len; self.step=step
        self.docs={}; self.windows=[]
        t0=time.time()
        for di,(did,text) in enumerate(text_map.items()):
            if gold_map is None:
                ids,offsets,exact=encode_with_char_offsets(text)
                tags=None; group=None; ranges=[]; stats=Counter()
            else:
                ids,offsets,tags,group,ranges,stats=tag_document(text,gold_map[did])
            ranges=sorted(ranges)                                   # (first_tok, last_tok, gold_idx), non-overlapping
            self.docs[did]=dict(text=text,ids=ids,offsets=offsets,tags=tags,group=group,ranges=ranges,
                                r_last=[r[1] for r in ranges],stats=stats)
            n=len(ids)
            if n==0: self.windows.append((did,0,0))
            else:
                for s in range(0,max(1,n),step):
                    e=min(n,s+seq_len); self.windows.append((did,s,e))
                    if e==n: break
            if (di+1)%100==0: print(f" tokenized {di+1}/{len(text_map)} docs",flush=True)
        print(f"Built {len(self.windows):,} windows from {len(self.docs):,} docs in {(time.time()-t0)/60:.1f} min")
    def __len__(self): return len(self.windows)
    def __getitem__(self,idx):
        did,s,e=self.windows[idx]; d=self.docs[did]; ids=d["ids"][s:e]
        item={"doc_id":did,"start_token":s,"end_token":e,"input_ids":torch.tensor(ids,dtype=torch.long)}
        if d["tags"] is not None:
            labels=list(d["tags"][s:e])
            # entities that are cut by the window border are ignored (no loss on their visible fragment)
            k=bisect.bisect_left(d["r_last"],s)
            while k<len(d["ranges"]):
                i,j,_=d["ranges"][k]
                if i>=e: break
                if i<s or j>=e:
                    for p in range(max(i,s),min(j,e-1)+1): labels[p-s]=IGNORE_INDEX
                k+=1
            item["labels"]=torch.tensor(labels,dtype=torch.long)
        return item

def collate_windows(batch):
    B=len(batch); T=max(max(1,len(x["input_ids"])) for x in batch)
    input_ids=torch.full((B,T),tokenizer.pad_token_id,dtype=torch.long)
    attention_mask=torch.zeros((B,T),dtype=torch.bool)
    labels=torch.full((B,T),IGNORE_INDEX,dtype=torch.long) if "labels" in batch[0] else None
    doc_ids=[]; starts=[]; ends=[]
    for i,x in enumerate(batch):
        n=len(x["input_ids"])
        if n:
            input_ids[i,:n]=x["input_ids"]; attention_mask[i,:n]=True
            if labels is not None: labels[i,:n]=x["labels"]
        doc_ids.append(x["doc_id"]); starts.append(x["start_token"]); ends.append(x["end_token"])
    out={"input_ids":input_ids,"attention_mask":attention_mask,"doc_ids":doc_ids,"start_tokens":starts,"end_tokens":ends}
    if labels is not None: out["labels"]=labels
    return out

train_ds=NERWindowDataset(fold1_text,fold1_gold)
val_ds=NERWindowDataset(fold2_text,fold2_gold)
print("Train windows:",len(train_ds))
print("Val windows:",len(val_ds))


Built 736 windows from 35 docs in 0.0 min
Built 748 windows from 35 docs in 0.0 min
Train windows: 736
Val windows: 748


In [9]:
# ===== Exact Phase-1 Model-1 architecture + LoRA =====
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model,dtype=torch.float32)
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids): return self.pe[position_ids]

class PrefixSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,attn_mask=None,past_kv=None,use_cache=False):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2); q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        if past_kv is not None:
            pk,pv=past_kv; k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,is_causal=False)
        y=y.transpose(1,2).contiguous().view(B,T,C); y=self.out_proj(y)
        return y,((k,v) if use_cache else None)

class EncoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__(); self.ln1=nn.LayerNorm(d_model)
        self.attn=PrefixSelfAttention(d_model,n_heads,dropout); self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(nn.Linear(d_model,ffn_dim),nn.GELU(),nn.Linear(ffn_dim,d_model),nn.Dropout(dropout))
        self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,attn_mask=None,past_kv=None,use_cache=False):
        a,new_kv=self.attn(self.ln1(x),attn_mask=attn_mask,past_kv=past_kv,use_cache=use_cache)
        x=x+self.resid_drop(a); x=x+self.ff(self.ln2(x)); return x,new_kv

class PrefixEncoderLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout,max_positions):
        super().__init__(); self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model); self.pos=SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([EncoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)])
        self.final_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        scaled_std=0.02/math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight,mean=0.0,std=scaled_std)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=scaled_std)
    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding): nn.init.normal_(m.weight,mean=0.0,std=0.02)
    @staticmethod
    def prefix_mask(T,prefix_len,device):
        q=torch.arange(T,device=device)[:,None]; k=torch.arange(T,device=device)[None,:]; p=int(prefix_len)
        allowed=((q<p)&(k<p))|((q>=p)&(k<=q)); return allowed[None,None,:,:]
    def forward_hidden(self,input_ids,prefix_len=None,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape; past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions: raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        x=self.token_emb(input_ids)+self.pos(position_ids).to(self.token_emb.weight.dtype); x=self.drop(x)
        if key_valid is not None: attn_mask=key_valid[:,None,None,:].bool()
        elif past_key_values is None and prefix_len is not None and int(prefix_len)<T:
            attn_mask=self.prefix_mask(T,prefix_len,input_ids.device)
        else: attn_mask=None
        new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,attn_mask=attn_mask,past_kv=past,use_cache=use_cache)
            if use_cache: new_cache.append(kv)
        return self.final_ln(x),new_cache
    def project(self,hidden): return F.linear(hidden,self.token_emb.weight)

class LoRALinear(nn.Module):
    """Frozen base Linear + trainable low-rank delta W = (alpha/r) * B@A."""
    def __init__(self,base,r=16,alpha=32,dropout=0.05):
        super().__init__(); assert isinstance(base,nn.Linear) and r>0
        self.base=base; self.r=int(r); self.alpha=float(alpha); self.scale=self.alpha/self.r
        self.lora_dropout=nn.Dropout(dropout)
        self.lora_A=nn.Parameter(torch.empty(self.r,base.in_features))
        self.lora_B=nn.Parameter(torch.zeros(base.out_features,self.r))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))
        for p in self.base.parameters(): p.requires_grad=False
    def forward(self,x):
        delta=F.linear(F.linear(self.lora_dropout(x),self.lora_A),self.lora_B)
        return self.base(x)+delta*self.scale

def freeze_backbone(backbone):
    for p in backbone.parameters(): p.requires_grad=False

def inject_lora(backbone):
    n=0
    for block in backbone.blocks:
        if LORA_TARGET_QKV:
            block.attn.qkv=LoRALinear(block.attn.qkv,LORA_R,LORA_ALPHA,LORA_DROPOUT); n+=1
        if LORA_TARGET_OUT:
            block.attn.out_proj=LoRALinear(block.attn.out_proj,LORA_R,LORA_ALPHA,LORA_DROPOUT); n+=1
        if LORA_TARGET_FFN:
            block.ff[0]=LoRALinear(block.ff[0],LORA_R,LORA_ALPHA,LORA_DROPOUT); n+=1
            block.ff[2]=LoRALinear(block.ff[2],LORA_R,LORA_ALPHA,LORA_DROPOUT); n+=1
    return n

class Model1ForNER(nn.Module):
    def __init__(self,backbone,n_tags,dropout=0.1):
        super().__init__(); self.backbone=backbone; self.ner_dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(backbone.d_model,n_tags)
    def forward(self,input_ids,attention_mask=None):
        # prefix_len=None => Model 1's native FULL bidirectional encoder attention.
        h,_=self.backbone.forward_hidden(input_ids,prefix_len=None,key_valid=attention_mask)
        return self.classifier(self.ner_dropout(h))

def trainable_state_dict(model):
    names={n for n,p in model.named_parameters() if p.requires_grad}
    sd=model.state_dict()
    return {k:v.detach().cpu() for k,v in sd.items() if k in names}

def load_trainable_state(model,state):
    own=model.state_dict()
    missing=[]
    for k,v in state.items():
        if k not in own: raise KeyError(f"Unexpected adapter key: {k}")
        own[k].copy_(v.to(own[k].device,dtype=own[k].dtype))
    expected={n for n,p in model.named_parameters() if p.requires_grad}
    missing=sorted(expected-set(state))
    if missing: raise KeyError(f"Missing trainable keys: {missing[:10]}")


In [10]:
phase1_obj=torch.load(MODEL1_BEST_PATH,map_location="cpu",weights_only=False)
assert "model" in phase1_obj,"Checkpoint must contain model state_dict"
cfg=phase1_obj.get("config",{})
print("Checkpoint step:",phase1_obj.get("global_step",phase1_obj.get("step")))
print("Checkpoint best_val:",phase1_obj.get("best_val",phase1_obj.get("val_loss")))
print("Config:",cfg)

# Model-1 notebook defaults; checkpoint config is authoritative when present.
V=int(cfg.get("vocab_size",24576)); D=int(cfg.get("d_model",768)); L=int(cfg.get("n_layers",14))
H=int(cfg.get("n_heads",12)); FF=int(cfg.get("ffn_dim",3072)); DROP=float(cfg.get("dropout",0.0))
MP=int(cfg.get("max_positions",2048))
assert V==len(tokenizer),f"checkpoint vocab={V}, tokenizer={len(tokenizer)}"
assert NER_SEQ_LEN<=MP

backbone=PrefixEncoderLM(V,D,L,H,FF,DROP,MP)
backbone.load_state_dict(phase1_obj["model"],strict=True)
print("✅ Phase-1 Model 1 loaded STRICTLY before LoRA injection")
freeze_backbone(backbone)
n_lora_modules=inject_lora(backbone)
model=Model1ForNER(backbone,N_TAGS,HEAD_DROPOUT).to(DEVICE)

# Only LoRA + head should train.
trainable_names=[n for n,p in model.named_parameters() if p.requires_grad]
assert trainable_names and all(("lora_A" in n or "lora_B" in n or n.startswith("classifier.")) for n in trainable_names)
total=sum(p.numel() for p in model.parameters()); trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print("LoRA modules injected:",n_lora_modules)
print(f"Total params: {total/1e6:.3f}M | trainable: {trainable/1e6:.3f}M ({100*trainable/total:.3f}%)")
print("Trainable tensors:",len(trainable_names))
print("First trainable names:",trainable_names[:8])


# ---- Extra v3 safety checks ----
assert V==24576 and D==768 and L==14 and H==12 and FF==3072, (
    f"Unexpected Model-1 architecture: V={V}, D={D}, L={L}, H={H}, FF={FF}"
)
assert n_lora_modules == L * (
    int(LORA_TARGET_QKV) + int(LORA_TARGET_OUT) + 2*int(LORA_TARGET_FFN)
), "Unexpected number of injected LoRA modules"

# LoRA-B is initialized to zero, so immediately after injection every LoRA delta must be exactly zero.
with torch.no_grad():
    max_abs_B=0.0
    for n,p in model.named_parameters():
        if "lora_B" in n:
            max_abs_B=max(max_abs_B,float(p.detach().abs().max().cpu()))
print("Initial max |LoRA B|:",max_abs_B)
assert max_abs_B==0.0, "LoRA B must start at zero so the adapted model initially equals Phase-1 Model 1."
print("✅ LoRA starts as an exact zero-delta adaptation of the Phase-1 backbone.")


Checkpoint step: 20332
Checkpoint best_val: 2.3875660663470626
Config: {'vocab_size': 24576, 'd_model': 768, 'n_layers': 14, 'n_heads': 12, 'ffn_dim': 3072, 'dropout': 0.0, 'max_positions': 2048, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384}
✅ Phase-1 Model 1 loaded STRICTLY before LoRA injection
LoRA modules injected: 56
Total params: 123.649M | trainable: 5.543M (4.483%)
Trainable tensors: 114
First trainable names: ['backbone.blocks.0.attn.qkv.lora_A', 'backbone.blocks.0.attn.qkv.lora_B', 'backbone.blocks.0.attn.out_proj.lora_A', 'backbone.blocks.0.attn.out_proj.lora_B', 'backbone.blocks.0.ff.0.lora_A', 'backbone.blocks.0.ff.0.lora_B', 'backbone.blocks.0.ff.2.lora_A', 'backbone.blocks.0.ff.2.lora_B']
Initial max |LoRA B|: 0.0
✅ LoRA starts as an exact zero-delta adaptation of the Phase-1 backbone.


In [11]:
# ===== CHECKPOINT/TOKENIZER SANITY CHECK BEFORE NER TRAINING =====
# Use the ORIGINAL Phase-1 tokenization (tokenizer.encode uses r"\S+|\s+") and prefix-LM objective.
@torch.no_grad()
def prefix_lm_loss_check(text_map,n_docs=TOKENIZER_CHECK_DOCS,max_tokens=768):
    # Temporarily evaluate the pristine frozen base path. LoRA B starts at zero, so it is identical at initialization.
    model.eval(); losses=[]
    docs=list(text_map.values())[:n_docs]
    for text in docs:
        ids=tokenizer.encode(text)[:max_tokens]
        if len(ids)<64: continue
        x=torch.tensor(ids,dtype=torch.long,device=DEVICE)[None,:]
        p=max(16,min(len(ids)-2,len(ids)//2))
        mask=torch.ones_like(x,dtype=torch.bool)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            h,_=model.backbone.forward_hidden(x,prefix_len=p,key_valid=None)
            lg=model.backbone.project(h[:,p-1:-1,:])
        target=x[:,p:]
        loss=F.cross_entropy(lg.float().reshape(-1,V),target.reshape(-1))
        losses.append(float(loss))
    return float(np.mean(losses)) if losses else float("nan")

chk=prefix_lm_loss_check(fold1_text)
print(f"Model-1 prefix-LM compatibility loss on Phase-2 text: {chk:.3f}")
if not np.isfinite(chk) or chk>8: print("❌ Red flag — inspect checkpoint/tokenizer/model definition before training.")
elif chk>5: print("⚠️ High distribution-shift loss; inspect before training.")
else: print("✅ Tokenizer + checkpoint + exact Model-1 definition look consistent.")


Model-1 prefix-LM compatibility loss on Phase-2 text: 3.450
✅ Tokenizer + checkpoint + exact Model-1 definition look consistent.


In [12]:
def count_training_tags(ds):
    c=Counter()
    for d in ds.docs.values():
        if d["tags"] is not None: c.update(d["tags"])
    return c

tag_counts=count_training_tags(train_ds)
if USE_CLASS_WEIGHTS:
    counts=np.array([max(1,tag_counts.get(i,0)) for i in range(N_TAGS)],dtype=np.float64)
    weights=np.sqrt(counts.sum()/counts); weights/=np.median(weights[1:]) if len(weights)>1 else 1.0
    weights=np.clip(weights,0.20,5.0); weights[O_ID]=min(weights[O_ID],0.35)
    CLASS_WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
else: CLASS_WEIGHTS=None

print("\nTraining tag statistics:")
for tid in range(N_TAGS):
    if tag_counts.get(tid,0)>0:
        w=float(CLASS_WEIGHTS[tid].detach().cpu()) if CLASS_WEIGHTS is not None else 1.0
        print(f"{ID2TAG[tid]:16s} count={tag_counts.get(tid,0):8d} weight={w:.3f}")

train_loader=DataLoader(train_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,collate_fn=collate_windows,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows,num_workers=2,pin_memory=True)

lora_params=[p for n,p in model.named_parameters() if p.requires_grad and ("lora_A" in n or "lora_B" in n)]
head_params=list(model.classifier.parameters())
assert sum(p.numel() for p in lora_params)>0 and sum(p.numel() for p in head_params)>0
optimizer=torch.optim.AdamW([
    {"params":lora_params,"lr":LORA_LR},
    {"params":head_params,"lr":HEAD_LR},
],weight_decay=WEIGHT_DECAY)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=max(1,updates_per_epoch*EPOCHS); warmup_updates=max(1,int(WARMUP_RATIO*total_updates))
def lr_lambda(step):
    if step<warmup_updates: return (step+1)/warmup_updates
    progress=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(1.0,progress)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
scaler=torch.amp.GradScaler("cuda",enabled=(AMP and DEVICE=="cuda"))
trainable_params=[p for p in model.parameters() if p.requires_grad]
print("Batches/epoch:",len(train_loader),"updates/epoch:",updates_per_epoch,"total updates:",total_updates)
print(f"LoRA trainable params: {sum(p.numel() for p in lora_params)/1e6:.3f}M | head: {sum(p.numel() for p in head_params)/1e6:.3f}M")



Training tag statistics:
O                count=  390766 weight=0.200
B-APP            count=     169 weight=1.015
I-APP            count=     536 weight=0.570
L-APP            count=     169 weight=1.015
U-APP            count=      34 weight=2.262
B-RESP           count=     174 weight=1.000
I-RESP           count=     498 weight=0.591
L-RESP           count=     174 weight=1.000
U-RESP           count=      17 weight=3.199
B-A.COUNSEL      count=      64 weight=1.649
I-A.COUNSEL      count=     275 weight=0.795
L-A.COUNSEL      count=      64 weight=1.649
U-A.COUNSEL      count=       6 weight=5.000
B-R.COUNSEL      count=      83 weight=1.448
I-R.COUNSEL      count=     295 weight=0.768
L-R.COUNSEL      count=      83 weight=1.448
U-R.COUNSEL      count=       4 weight=5.000
B-JUDGE          count=     118 weight=1.214
I-JUDGE          count=     574 weight=0.551
L-JUDGE          count=     118 weight=1.214
U-JUDGE          count=       4 weight=5.000
B-WIT            count=      

In [13]:
def strict_scores(truth,pred,labels=LABELS,verbose=True):
    if set(truth)!=set(pred): raise ValueError("ID mismatch")
    stats={l:[0,0,0] for l in labels}; allowed=set(labels)
    def ss(xs):
        out=set()
        for s in xs or []:
            a,b,l=s[:3]
            if not isinstance(a,int) or isinstance(a,bool) or not isinstance(b,int) or isinstance(b,bool): raise ValueError("bad offsets")
            if a<0 or b<=a or l not in allowed: raise ValueError("bad span")
            t=(a,b,l)
            if t in out: raise ValueError("duplicate span")
            out.add(t)
        return out
    for did in truth:
        g=ss(truth[did]); p=ss(pred[did])
        for l in labels:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    ps=[]; rs=[]; fs=[]; per={}
    for l in labels:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        f=2*pr*rc/(pr+rc) if pr+rc else 0
        ps.append(pr); rs.append(rc); fs.append(f); per[l]=(pr,rc,f,tp,fp,fn)
        if verbose: print(f"{l:10s}: P={pr:.6f} R={rc:.6f} F1={f:.6f}")
    out={"macro_f1":float(np.mean(fs)),"macro_precision":float(np.mean(ps)),"macro_recall":float(np.mean(rs)),"per_label":per}
    if verbose:
        print(f"Strict Macro-F1:        {out['macro_f1']:.6f}")
        print(f"Strict Macro-Precision: {out['macro_precision']:.6f}")
        print(f"Strict Macro-Recall:    {out['macro_recall']:.6f}")
    return out

def gold_for_eval(gold_map):
    out={}
    for did,ents in gold_map.items():
        seen=set(); lst=[]
        for a,b,l,et in ents:
            t=(int(a),int(b),str(l))
            if t not in seen: seen.add(t); lst.append(list(t))      # evaluator rejects duplicate spans
        out[did]=lst
    return out

# ---- constrained BILOU Viterbi ----
def _build_transitions():
    pref=[None]+[t.split("-",1)[0] for t in TAG_NAMES[1:]]
    lab=[None]+[t.split("-",1)[1] for t in TAG_NAMES[1:]]
    allow=np.zeros((N_TAGS,N_TAGS),dtype=bool)
    for a in range(N_TAGS):
        a_open = pref[a] in ("B","I")
        for b in range(N_TAGS):
            if a_open: allow[a,b]=(pref[b] in ("I","L") and lab[b]==lab[a])
            else:      allow[a,b]=(b==O_ID or pref[b] in ("B","U"))
    start=np.array([ (i==O_ID or pref[i] in ("B","U")) for i in range(N_TAGS)])
    end=np.array([ (i==O_ID or pref[i] in ("L","U")) for i in range(N_TAGS)])
    return allow,start,end
TRANS_ALLOW,START_OK,END_OK=_build_transitions()
NEG=-1e9

def softmax_np(x):
    x=x-x.max(-1,keepdims=True); e=np.exp(x); return e/e.sum(-1,keepdims=True)

def viterbi_path(logp):
    n=logp.shape[0]
    trans=np.where(TRANS_ALLOW,0.0,NEG).astype(np.float32)
    score=np.where(START_OK,logp[0],NEG).astype(np.float32)
    back=np.zeros((n,N_TAGS),dtype=np.int16)
    for t in range(1,n):
        cand=score[:,None]+trans                     # (from,to)
        bp=cand.argmax(0); score=cand[bp,np.arange(N_TAGS)]+logp[t]; back[t]=bp
    score=np.where(END_OK,score,NEG)
    path=np.zeros(n,dtype=np.int64); path[-1]=int(score.argmax())
    for t in range(n-1,0,-1): path[t-1]=back[t,path[t]]
    return path

def decode_spans(doc,logits,viterbi=None):
    """Returns [(start_char,end_char,label,confidence)], confidence = min token prob of the span's tags."""
    if len(logits)==0: return []
    viterbi=USE_VITERBI if viterbi is None else viterbi
    logits=np.asarray(logits,dtype=np.float32); probs=softmax_np(logits)
    if viterbi: pred=viterbi_path(np.log(probs+1e-9))
    else:       pred=probs.argmax(-1)
    conf=probs[np.arange(len(pred)),pred]; offsets=doc["offsets"]; text=doc["text"]; out=[]; i=0; n=len(pred)
    while i<n:
        tag=ID2TAG[int(pred[i])]
        if tag=="O": i+=1; continue
        pf,lab=tag.split("-",1)
        if pf=="U": j=i
        elif pf=="B":
            j=i+1
            while j<n and ID2TAG[int(pred[j])]==f"I-{lab}": j+=1
            if not(j<n and ID2TAG[int(pred[j])]==f"L-{lab}"): i+=1; continue     # malformed (argmax mode only)
        else: i+=1; continue
        a=offsets[i][0]; b=offsets[j][1]
        if 0<=a<b<=len(text) and text[a:b].strip(): out.append((int(a),int(b),lab,float(conf[i:j+1].min())))
        i=j+1
    return out

def _threshold_for_label(threshold,label):
    if isinstance(threshold,dict):
        return float(threshold.get(label,0.0))
    return float(threshold)

def filter_spans(spans,threshold=0.0):
    """threshold may be a scalar or {label: threshold} dict."""
    seen=set(); res=[]
    for a,b,l,c in spans:
        th=_threshold_for_label(threshold,l)
        if c>=th and (a,b,l) not in seen:
            seen.add((a,b,l)); res.append([a,b,l])
    return res

def decode_bilou_from_logits(doc,logits,threshold=0.0,viterbi=None):
    return filter_spans(decode_spans(doc,logits,viterbi),threshold)

def tune_global_threshold(span_cache,truth,grid):
    rows=[]
    for th in grid:
        pred={did:filter_spans(span_cache[did],th) for did in span_cache}
        sc=strict_scores(truth,pred,verbose=False)
        rows.append((sc["macro_f1"],float(th),sc))
    rows.sort(reverse=True,key=lambda x:x[0])
    return rows[0],rows

def tune_per_label_thresholds(span_cache,truth,grid):
    """
    Macro-F1 = mean of per-label F1; for a fixed decoded span set each label's threshold only affects that label.
    v2: the F1-vs-threshold curve is smoothed ([.25,.5,.25]) before the argmax so a single lucky threshold on a tiny
    held-out fold does not get picked (plateau centre instead of spike).
    """
    thresholds={}; details={}; grid=[float(x) for x in grid]
    for lab in LABELS:
        truth_l={did:[s for s in truth[did] if s[2]==lab] for did in truth}
        f1s=[]; scs=[]
        for th in grid:
            pred={did:[[a,b,l] for a,b,l,c in spans if l==lab and c>=th] for did,spans in span_cache.items()}
            sc=strict_scores(truth_l,pred,labels=[lab],verbose=False)
            f1s.append(sc["macro_f1"]); scs.append(sc)
        f=np.array(f1s)
        if THRESHOLD_SMOOTH and len(f)>=3:
            pad=np.concatenate([[f[0]],f,[f[-1]]]); f=0.25*pad[:-2]+0.5*pad[1:-1]+0.25*pad[2:]
        k=int(np.argmax(f)); thresholds[lab]=grid[k]; details[lab]=scs[k]
    combined={did:filter_spans(span_cache[did],thresholds) for did in span_cache}
    combined_sc=strict_scores(truth,combined,verbose=False)
    return thresholds,combined_sc,details


In [14]:
# ===== ORACLE CEILING TEST: gold tags -> decode -> strict F1 (no model involved) =====
# This is the best Strict Macro-F1 your tokenizer/alignment can possibly give on Fold 2.
# If it is far below ~0.95, token boundaries do not line up with gold spans (try PRESPLIT_PUNCT=True and compare).
def oracle_logits(doc):
    lg=np.full((len(doc["ids"]),N_TAGS),-10.0,dtype=np.float32)
    for k,t in enumerate(doc["tags"]): lg[k,t]=10.0
    return lg
oracle_pred={did:decode_bilou_from_logits(d,oracle_logits(d),0.0) for did,d in val_ds.docs.items()}
_truth_val=gold_for_eval(fold2_gold)
print("ORACLE (Fold 2):"); oracle_sc=strict_scores(_truth_val,oracle_pred,verbose=True)


ORACLE (Fold 2):
APP       : P=1.000000 R=0.995000 F1=0.997494
RESP      : P=1.000000 R=1.000000 F1=1.000000
A.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
R.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
JUDGE     : P=1.000000 R=1.000000 F1=1.000000
WIT       : P=1.000000 R=1.000000 F1=1.000000
AUTH      : P=1.000000 R=1.000000 F1=1.000000
COURT     : P=1.000000 R=0.980287 F1=0.990045
STAT      : P=1.000000 R=0.994558 F1=0.997271
PREC      : P=1.000000 R=1.000000 F1=1.000000
DATE      : P=1.000000 R=1.000000 F1=1.000000
CASENO    : P=1.000000 R=1.000000 F1=1.000000
Strict Macro-F1:        0.998734
Strict Macro-Precision: 1.000000
Strict Macro-Recall:    0.997487


In [15]:
@torch.no_grad()
def infer_dataset(model,ds,loader,threshold=0.0,return_logits=False):
    model.eval()
    doc_logits={did:np.zeros((len(d["ids"]),N_TAGS),dtype=np.float32) for did,d in ds.docs.items()}
    best_dist={did:np.full(len(d["ids"]),-1.0,dtype=np.float32) for did,d in ds.docs.items()}   # centrality of the chosen logits
    for bi,batch in enumerate(loader):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            logits=model(ids,mask)
        logits=logits.float().cpu().numpy()
        for r,did in enumerate(batch["doc_ids"]):
            s=batch["start_tokens"][r]; e=batch["end_tokens"][r]; n=len(ds.docs[did]["ids"])
            if e<=s: continue
            L=e-s; loc=np.arange(L)
            dl=loc if s>0 else np.full(L,1e9); dr=(L-1-loc) if e<n else np.full(L,1e9)   # no penalty at true doc borders
            dist=np.minimum(dl,dr).astype(np.float32)
            upd=dist>best_dist[did][s:e]
            doc_logits[did][s:e][upd]=logits[r,:L][upd]; best_dist[did][s:e][upd]=dist[upd]
        if (bi+1)%50==0: print(f"inference batch {bi+1}/{len(loader)}",flush=True)
    for did in best_dist:
        if len(best_dist[did]) and (best_dist[did]<0).any(): raise RuntimeError(f"{did}: unfilled tokens")
    pred={did:decode_bilou_from_logits(ds.docs[did],doc_logits[did],threshold) for did in ds.docs}
    return (pred,doc_logits) if return_logits else pred


In [17]:
def compute_loss(logits,labels):
    return F.cross_entropy(logits.float().view(-1,N_TAGS),labels.view(-1),weight=CLASS_WEIGHTS,ignore_index=IGNORE_INDEX)

print("🔥 ACTUAL LoRA FINE-TUNING STARTS HERE")
best_f1=-1.0; best_epoch=None; global_update=0; amp_skipped_updates=0; history=[]
truth_val=gold_for_eval(fold2_gold)

for epoch in range(1,EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True); running=0.0; n_batches=0; t0=time.time()
    for step,batch in enumerate(train_loader,1):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True); mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        y=batch["labels"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            logits=model(ids,mask); loss=compute_loss(logits,y)/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward(); running+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS; n_batches+=1
        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(trainable_params,MAX_GRAD_NORM)
            scale_before=scaler.get_scale(); scaler.step(optimizer); scaler.update(); scale_after=scaler.get_scale()
            optimizer.zero_grad(set_to_none=True)
            # AMP can skip optimizer.step on overflow; do not advance scheduler in that case.
            if scale_after>=scale_before:
                scheduler.step(); global_update+=1
            else:
                amp_skipped_updates+=1
                if amp_skipped_updates<=5:
                    print(f"⚠️ AMP overflow: optimizer/scheduler update skipped (count={amp_skipped_updates})")
        if step%100==0 or step==len(train_loader):
            print(f"epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} loss={running/max(1,n_batches):.4f} "
                  f"lora_lr={optimizer.param_groups[0]['lr']:.2e} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    _,val_logits=infer_dataset(model,val_ds,val_loader,0.0,True)
    span_cache={did:decode_spans(val_ds.docs[did],val_logits[did]) for did in val_ds.docs}
    (global_f1,global_th,global_sc),_=tune_global_threshold(span_cache,truth_val,THRESHOLD_GRID)
    if USE_PER_LABEL_THRESHOLDS:
        epoch_th,epoch_sc,_=tune_per_label_thresholds(span_cache,truth_val,THRESHOLD_GRID)
        epoch_f1=epoch_sc["macro_f1"]; threshold_desc="per-label"
    else:
        epoch_th=float(global_th); epoch_sc=global_sc; epoch_f1=global_f1; threshold_desc=f"global={global_th:.2f}"

    print(f"Epoch {epoch}: global-threshold Macro-F1={global_f1:.6f} @ {global_th:.2f} | "
          f"selected({threshold_desc}) Macro-F1={epoch_f1:.6f} P={epoch_sc['macro_precision']:.6f} R={epoch_sc['macro_recall']:.6f}")
    if isinstance(epoch_th,dict): print("   per-label thresholds:"," ".join(f"{l}={epoch_th[l]:.2f}" for l in LABELS))
    history.append({"epoch":epoch,"train_loss":running/max(1,n_batches),"threshold":epoch_th,
                    "global_threshold":float(global_th),"global_macro_f1":float(global_f1),
                    "macro_f1":epoch_f1,"macro_precision":epoch_sc["macro_precision"],"macro_recall":epoch_sc["macro_recall"]})
    if epoch_f1>best_f1:
        best_f1=epoch_f1; best_epoch=epoch
        torch.save({
            "trainable_state":trainable_state_dict(model),"phase1_checkpoint":MODEL1_BEST_PATH,"phase1_config":cfg,
            "tag_names":TAG_NAMES,"labels":LABELS,"best_epoch":best_epoch,"best_threshold":epoch_th,
            "strict_macro_f1":epoch_f1,"history":history,
            "experiment":"Phase2 Exp2 Model1 LoRA qkv+out+ffn BILOU","ner_seq_len":NER_SEQ_LEN,"window_overlap":WINDOW_OVERLAP,
            "lora_r":LORA_R,"lora_alpha":LORA_ALPHA,"lora_dropout":LORA_DROPOUT,
            "viterbi":bool(USE_VITERBI),"presplit_punct":bool(PRESPLIT_PUNCT),
            "use_per_label_thresholds":bool(USE_PER_LABEL_THRESHOLDS),"threshold_grid":THRESHOLD_GRID,
            "tokenizer_sha256":tok_sha},BEST_NER_PATH)
        print("✅ New best LoRA adapter saved:",BEST_NER_PATH)
        print("   per-label F1:"," ".join(f"{l}={v[2]:.3f}" for l,v in epoch_sc["per_label"].items()))
    print("-"*90)

print("Best epoch:",best_epoch,"Best held-out Strict Macro-F1:",best_f1)
print("AMP-skipped optimizer updates:",amp_skipped_updates)
for x in history: print(x)


🔥 ACTUAL LoRA FINE-TUNING STARTS HERE
epoch 1/12 step 100/368 loss=0.7858 lora_lr=2.64e-04 elapsed=0.2m
epoch 1/12 step 200/368 loss=0.6300 lora_lr=3.82e-04 elapsed=0.3m
epoch 1/12 step 300/368 loss=0.5447 lora_lr=4.91e-04 elapsed=0.5m
epoch 1/12 step 368/368 loss=0.5029 lora_lr=5.00e-04 elapsed=0.6m
inference batch 50/187
inference batch 100/187
inference batch 150/187
Epoch 1: global-threshold Macro-F1=0.263672 @ 0.00 | selected(per-label) Macro-F1=0.267853 P=0.305244 R=0.249405
   per-label thresholds: APP=0.00 RESP=0.00 A.COUNSEL=0.00 R.COUNSEL=0.00 JUDGE=0.20 WIT=0.10 AUTH=0.25 COURT=0.20 STAT=0.10 PREC=0.00 DATE=0.00 CASENO=0.00
✅ New best LoRA adapter saved: /kaggle/working/phase2_exp2_model1_lora/best_lora_ner.pt
   per-label F1: APP=0.025 RESP=0.000 A.COUNSEL=0.186 R.COUNSEL=0.097 JUDGE=0.180 WIT=0.036 AUTH=0.284 COURT=0.744 STAT=0.438 PREC=0.189 DATE=0.836 CASENO=0.200
------------------------------------------------------------------------------------------
epoch 2/12 step 1

In [18]:
def fresh_phase1_lora_ner_model():
    bb=PrefixEncoderLM(V,D,L,H,FF,DROP,MP)
    bb.load_state_dict(phase1_obj["model"],strict=True)
    freeze_backbone(bb); inject_lora(bb)
    return Model1ForNER(bb,N_TAGS,HEAD_DROPOUT).to(DEVICE)

best_obj=torch.load(BEST_NER_PATH,map_location="cpu",weights_only=False)
model=fresh_phase1_lora_ner_model()
load_trainable_state(model,best_obj["trainable_state"])
BEST_THRESHOLD=best_obj["best_threshold"]; BEST_EPOCH=int(best_obj["best_epoch"])
val_pred=infer_dataset(model,val_ds,val_loader,BEST_THRESHOLD)
val_score=strict_scores(truth_val,val_pred,verbose=True)
with open(VAL_PRED_PATH,"w",encoding="utf-8") as f: json.dump(val_pred,f,ensure_ascii=False,indent=2)
print("Best epoch:",BEST_EPOCH)
print("Best threshold(s):",BEST_THRESHOLD)
print("Saved:",VAL_PRED_PATH)

LOCAL_RESULT={
    "experiment":"Phase2 Exp2 Model1 LoRA qkv+out+ffn BILOU",
    "best_epoch":BEST_EPOCH,
    "threshold":BEST_THRESHOLD,
    "strict_macro_f1":float(val_score["macro_f1"]),
    "strict_macro_precision":float(val_score["macro_precision"]),
    "strict_macro_recall":float(val_score["macro_recall"]),
}
( ROOT/"heldout_result.json" ).write_text(json.dumps(LOCAL_RESULT,indent=2),encoding="utf-8")
print("Saved held-out result summary:",ROOT/"heldout_result.json")


inference batch 50/187
inference batch 100/187
inference batch 150/187
APP       : P=0.103659 R=0.085000 F1=0.093407
RESP      : P=0.298701 R=0.121053 F1=0.172285
A.COUNSEL : P=0.327586 R=0.304000 F1=0.315353
R.COUNSEL : P=0.347826 R=0.329897 F1=0.338624
JUDGE     : P=0.381944 R=0.450820 F1=0.413534
WIT       : P=0.350000 R=0.058333 F1=0.100000
AUTH      : P=0.348371 R=0.297009 F1=0.320646
COURT     : P=0.812825 R=0.840502 F1=0.826432
STAT      : P=0.705706 R=0.639456 F1=0.670949
PREC      : P=0.357798 R=0.348837 F1=0.353261
DATE      : P=0.969058 R=0.904199 F1=0.935506
CASENO    : P=0.607438 R=0.436202 F1=0.507772
Strict Macro-F1:        0.420647
Strict Macro-Precision: 0.467576
Strict Macro-Recall:    0.401276
Best epoch: 7
Best threshold(s): {'APP': 0.0, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.1, 'JUDGE': 0.05, 'WIT': 0.2, 'AUTH': 0.1, 'COURT': 0.1, 'STAT': 0.05, 'PREC': 0.05, 'DATE': 0.3, 'CASENO': 0.1}
Saved: /kaggle/working/phase2_exp2_model1_lora/val_prediction.json
Saved 

## Final refit on all labelled data

After Fold 2 selects the LoRA epoch count and confidence thresholds, the final refit starts again from the untouched Phase-1 Model-1 checkpoint and trains a **fresh LoRA adapter** on Fold 1 + Fold 2. The base model remains frozen.


In [20]:
if not DO_FINAL_REFIT_ON_ALL:
    print("Skipping final refit. Using held-out-selected LoRA model for test inference.")
    final_model=model
else:
    # Fresh deterministic final refit; model is recreated from untouched Phase-1 weights.
    random.seed(SEED+1); np.random.seed(SEED+1); torch.manual_seed(SEED+1)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED+1)
    all_text=dict(fold1_text); all_text.update(fold2_text)
    all_gold=dict(fold1_gold); all_gold.update(fold2_gold)
    all_ds=NERWindowDataset(all_text,all_gold)
    all_loader=DataLoader(all_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,collate_fn=collate_windows,num_workers=2,pin_memory=True)
    final_model=fresh_phase1_lora_ner_model()

    all_counts=count_training_tags(all_ds)
    if USE_CLASS_WEIGHTS:
        counts=np.array([max(1,all_counts.get(i,0)) for i in range(N_TAGS)],dtype=np.float64)
        w=np.sqrt(counts.sum()/counts); w/=np.median(w[1:]); w=np.clip(w,0.20,5.0); w[O_ID]=min(w[O_ID],0.35)
        FW=torch.tensor(w,dtype=torch.float32,device=DEVICE)
    else: FW=None

    lp=[p for n,p in final_model.named_parameters() if p.requires_grad and ("lora_A" in n or "lora_B" in n)]
    hp=list(final_model.classifier.parameters()); tp=[p for p in final_model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW([{"params":lp,"lr":LORA_LR},{"params":hp,"lr":HEAD_LR}],weight_decay=WEIGHT_DECAY)
    upe=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS); total=max(1,upe*BEST_EPOCH); warm=max(1,int(WARMUP_RATIO*total))
    def flr(step):
        if step<warm: return (step+1)/warm
        p=(step-warm)/max(1,total-warm); return 0.5*(1+math.cos(math.pi*min(1.0,p)))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,flr)
    scl=torch.amp.GradScaler("cuda",enabled=(AMP and DEVICE=="cuda"))

    refit_amp_skips=0
    for ep in range(1,BEST_EPOCH+1):
        final_model.train(); opt.zero_grad(set_to_none=True); run=0.0; nb=0; t0=time.time()
        for step,batch in enumerate(all_loader,1):
            ids=batch["input_ids"].to(DEVICE,non_blocking=True); mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
            y=batch["labels"].to(DEVICE,non_blocking=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
                lg=final_model(ids,mask)
                loss=F.cross_entropy(lg.float().view(-1,N_TAGS),y.view(-1),weight=FW,ignore_index=IGNORE_INDEX)/GRAD_ACCUM_STEPS
            scl.scale(loss).backward(); run+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS; nb+=1
            if step%GRAD_ACCUM_STEPS==0 or step==len(all_loader):
                scl.unscale_(opt); torch.nn.utils.clip_grad_norm_(tp,MAX_GRAD_NORM)
                sb=scl.get_scale(); scl.step(opt); scl.update(); sa=scl.get_scale(); opt.zero_grad(set_to_none=True)
                if sa>=sb:
                    sched.step()
                else:
                    refit_amp_skips+=1
                    if refit_amp_skips<=5:
                        print(f"⚠️ Refit AMP overflow: optimizer/scheduler update skipped (count={refit_amp_skips})")
            if step%100==0 or step==len(all_loader):
                print(f"refit epoch {ep}/{BEST_EPOCH} step {step}/{len(all_loader)} loss={run/max(1,nb):.4f} elapsed={(time.time()-t0)/60:.1f}m")

    torch.save({"trainable_state":trainable_state_dict(final_model),"phase1_config":cfg,"tag_names":TAG_NAMES,"labels":LABELS,
                "epochs":BEST_EPOCH,"threshold":BEST_THRESHOLD,
                "experiment":"Phase2 Exp2 Model1 LoRA qkv+out+ffn BILOU final refit",
                "lora_r":LORA_R,"lora_alpha":LORA_ALPHA,"lora_dropout":LORA_DROPOUT,
                "viterbi":bool(USE_VITERBI),"presplit_punct":bool(PRESPLIT_PUNCT),
                "use_per_label_thresholds":bool(USE_PER_LABEL_THRESHOLDS),"tokenizer_sha256":tok_sha},FINAL_REFIT_PATH)
    print("✅ Final LoRA refit adapter saved:",FINAL_REFIT_PATH)
    print("Refit AMP-skipped optimizer updates:",refit_amp_skips)


Built 1,484 windows from 70 docs in 0.0 min
refit epoch 1/7 step 100/742 loss=3.3928 elapsed=0.2m
refit epoch 1/7 step 200/742 loss=2.1525 elapsed=0.3m
refit epoch 1/7 step 300/742 loss=1.6391 elapsed=0.5m
refit epoch 1/7 step 400/742 loss=1.3409 elapsed=0.7m
refit epoch 1/7 step 500/742 loss=1.1422 elapsed=0.9m
refit epoch 1/7 step 600/742 loss=0.9980 elapsed=1.0m
refit epoch 1/7 step 700/742 loss=0.8957 elapsed=1.2m
refit epoch 1/7 step 742/742 loss=0.8588 elapsed=1.3m
refit epoch 2/7 step 100/742 loss=0.2074 elapsed=0.2m
refit epoch 2/7 step 200/742 loss=0.2210 elapsed=0.3m
refit epoch 2/7 step 300/742 loss=0.2153 elapsed=0.5m
refit epoch 2/7 step 400/742 loss=0.2076 elapsed=0.7m
refit epoch 2/7 step 500/742 loss=0.2065 elapsed=0.9m
refit epoch 2/7 step 600/742 loss=0.2049 elapsed=1.0m
refit epoch 2/7 step 700/742 loss=0.2028 elapsed=1.2m
refit epoch 2/7 step 742/742 loss=0.2000 elapsed=1.3m
refit epoch 3/7 step 100/742 loss=0.1660 elapsed=0.2m
refit epoch 3/7 step 200/742 loss=0.16

In [21]:
test_ds=NERWindowDataset(test_text,gold_map=None)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows,num_workers=2,pin_memory=True)

def validate_prediction(pred,text_map,labels):
    if not isinstance(pred,dict) or not pred: raise ValueError("prediction must be non-empty dict")
    if set(pred)!=set(text_map): raise ValueError(f"ID mismatch missing={len(set(text_map)-set(pred))} extra={len(set(pred)-set(text_map))}")
    allowed=set(labels); n=0
    for did,ents in pred.items():
        if not isinstance(ents,list): raise ValueError(f"{did}: entities must be list")
        seen=set(); text=text_map[did]
        for j,s in enumerate(ents):
            if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f"{did} entity {j}: bad format")
            a,b,l=s[:3]
            if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError("offsets must be ints")
            if not (0<=a<b<=len(text)): raise ValueError(f"{did}: invalid span {a,b}")
            if l not in allowed: raise ValueError(f"{did}: invalid label {l}")
            t=(a,b,l)
            if t in seen: raise ValueError(f"{did}: duplicate {t}")
            seen.add(t); n+=1
    return n

def write_submission(pred,json_path,zip_path):
    n=validate_prediction(pred,test_text,LABELS)
    with open(json_path,"w",encoding="utf-8") as f: json.dump(pred,f,ensure_ascii=False)
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf: zf.write(json_path,arcname="prediction.json")
    with zipfile.ZipFile(zip_path,"r") as zf:
        names=zf.namelist(); assert names==["prediction.json"]
        validate_prediction(json.loads(zf.read("prediction.json").decode("utf-8")),test_text,LABELS)
    print(f"✅ {zip_path.name}: docs={len(pred)} spans={n} zip={names}")
    return n

variants={"heldout":model}
if final_model is not model: variants["refit"]=final_model
for name,m in variants.items():
    pred=infer_dataset(m,test_ds,test_loader,BEST_THRESHOLD)
    write_submission(pred,ROOT/f"prediction_{name}.json",ROOT/f"submission_{name}.zip")
    if m is final_model:                      # primary files = the model chosen for the final submission
        submission_pred=pred; n_spans=write_submission(pred,TEST_PRED_PATH,SUBMISSION_ZIP)
print("Recommended: submit BOTH submission_heldout.zip and submission_refit.zip if submission slots allow; both belong to this same LoRA experiment. Keep the better leaderboard result as Exp2.")
print("✅ PHASE-2 SUBMISSION(S) PASSED ALL STRUCTURAL CHECKS")


Built 789 windows from 35 docs in 0.0 min
inference batch 50/198
inference batch 100/198
inference batch 150/198
✅ submission_heldout.zip: docs=35 spans=3931 zip=['prediction.json']
inference batch 50/198
inference batch 100/198
inference batch 150/198
✅ submission_refit.zip: docs=35 spans=4148 zip=['prediction.json']
✅ submission.zip: docs=35 spans=4148 zip=['prediction.json']
Recommended: submit BOTH submission_heldout.zip and submission_refit.zip if submission slots allow; both belong to this same LoRA experiment. Keep the better leaderboard result as Exp2.
✅ PHASE-2 SUBMISSION(S) PASSED ALL STRUCTURAL CHECKS


In [22]:
EVALUATOR='#!/usr/bin/env python3\n"""Local evaluator for Phase 2.\n\nUsage:\n  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt\n"""\nimport argparse, json, sys\n\nclass DuplicateJSONKeyError(ValueError): pass\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\ndef pairs_no_duplicates(pairs):\n    out={}; dup=[]\n    for k,v in pairs:\n        if k in out: dup.append(k)\n        out[k]=v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\ndef load_json(path):\n    with open(path,encoding=\'utf-8\') as f:\n        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\ndef labels(path):\n    with open(path,encoding=\'utf-8\') as f: xs=[x.strip() for x in f if x.strip()]\n    if not xs: raise ValueError(\'labels.txt is empty.\')\n    if len(xs)!=12: raise ValueError(f\'Expected 12 labels, found {len(xs)}.\')\n    if len(set(xs))!=len(xs): raise ValueError(\'labels.txt contains duplicate labels.\')\n    return xs\ndef spans(x, allowed, where):\n    if x is None: return set()\n    if not isinstance(x,list): raise ValueError(f\'{where}: expected a list of entities.\')\n    out=set()\n    for j,s in enumerate(x):\n        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f\'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].\')\n        a,b,l=s[:3]\n        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f\'{where}, entity {j}: start/end must be integers.\')\n        if a<0 or b<=a: raise ValueError(f\'{where}, entity {j}: require 0 <= start < end.\')\n        if not isinstance(l,str) or l not in allowed: raise ValueError(f\'{where}, entity {j}: invalid label {l!r}.\')\n        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f\'{where}, entity {j}: entity_text must be a string.\')\n        t=(a,b,l)\n        if t in out: raise ValueError(f\'{where}, entity {j}: duplicate span {t}.\')\n        out.add(t)\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 2 entity predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); ap.add_argument(\'--labels\',required=True)\n    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)\n    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError(\'Truth and prediction must be non-empty JSON objects.\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    stats={l:[0,0,0] for l in labs}; allowed=set(labs)\n    for did in truth:\n        g=spans(truth[did],allowed,f\'truth[{did}]\'); p=spans(pred[did],allowed,f\'prediction[{did}]\')\n        for l in labs:\n            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}\n            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)\n    precisions=[]; recalls=[]; f1s=[]\n    print(f\'Examples: {len(truth)}\')\n    for l in labs:\n        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0\n        precisions.append(pr); recalls.append(rc); f1s.append(f)\n        print(f\'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}\')\n    print(f\'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}\')\n    print(f\'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}\')\n    print(f\'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
eval_path=EVAL_SCRIPT_PATH
eval_path.write_text(EVALUATOR,encoding="utf-8")
print("Evaluator written:",eval_path)


print("\nRunning standalone local evaluator on held-out Fold 2...")
cmd=[sys.executable,str(EVAL_SCRIPT_PATH),"--truth",phase2_paths["fold2_gold"],"--prediction",str(VAL_PRED_PATH),"--labels",phase2_paths["labels"]]
res=subprocess.run(cmd,text=True,capture_output=True)
print(res.stdout)
if res.stderr: print("STDERR:\n"+res.stderr)
assert res.returncode==0,"Standalone evaluator failed"
print("✅ Standalone evaluator completed successfully.")


Evaluator written: /kaggle/working/phase2_exp2_model1_lora/evaluate_phase2.py

Running standalone local evaluator on held-out Fold 2...
Examples: 35
APP: P=0.103659 R=0.085000 F1=0.093407
RESP: P=0.298701 R=0.121053 F1=0.172285
A.COUNSEL: P=0.327586 R=0.304000 F1=0.315353
R.COUNSEL: P=0.347826 R=0.329897 F1=0.338624
JUDGE: P=0.381944 R=0.450820 F1=0.413534
WIT: P=0.350000 R=0.058333 F1=0.100000
AUTH: P=0.348371 R=0.297009 F1=0.320646
COURT: P=0.812825 R=0.840502 F1=0.826432
STAT: P=0.705706 R=0.639456 F1=0.670949
PREC: P=0.357798 R=0.348837 F1=0.353261
DATE: P=0.969058 R=0.904199 F1=0.935506
CASENO: P=0.607438 R=0.436202 F1=0.507772
Strict Macro-F1:        0.420647
Strict Macro-Precision: 0.467576
Strict Macro-Recall:    0.401276

✅ Standalone evaluator completed successfully.


In [23]:
print("KEEP THESE FILES:")
print(" Phase-1 Model-1 best checkpoint:",MODEL1_BEST_PATH)
print(" Exact tokenizer:",TOKENIZER_PATH)
print(" Phase-2 best LoRA heldout adapter:",BEST_NER_PATH)
if FINAL_REFIT_PATH.exists(): print(" Phase-2 final LoRA adapter:",FINAL_REFIT_PATH)
if VAL_PRED_PATH.exists(): print(" held-out val_prediction.json:",VAL_PRED_PATH)
if (ROOT/"heldout_result.json").exists(): print(" held-out result summary:",ROOT/"heldout_result.json")
if TEST_PRED_PATH.exists(): print(" prediction.json:",TEST_PRED_PATH)
if SUBMISSION_ZIP.exists(): print(" submission.zip:",SUBMISSION_ZIP)

print("\nExperiment identity: Model1 + LoRA(qkv+out+ffn) + BILOU token head")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print("Base Phase-1 weights frozen; only LoRA matrices + classifier train.")
print("Tokenizer SHA-256:",tok_sha)
if "BEST_EPOCH" in globals(): print("Best held-out epoch:",BEST_EPOCH)
if "BEST_THRESHOLD" in globals(): print("Selected confidence threshold(s):",BEST_THRESHOLD)

print("Experiment status: ONE distinct Phase-2 experiment = Model1 + LoRA. Hyperparameter/submission variants do not count as extra techniques.")


KEEP THESE FILES:
 Phase-1 Model-1 best checkpoint: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
 Exact tokenizer: /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
 Phase-2 best LoRA heldout adapter: /kaggle/working/phase2_exp2_model1_lora/best_lora_ner.pt
 Phase-2 final LoRA adapter: /kaggle/working/phase2_exp2_model1_lora/final_refit_lora_ner.pt
 held-out val_prediction.json: /kaggle/working/phase2_exp2_model1_lora/val_prediction.json
 held-out result summary: /kaggle/working/phase2_exp2_model1_lora/heldout_result.json
 prediction.json: /kaggle/working/phase2_exp2_model1_lora/prediction.json
 submission.zip: /kaggle/working/phase2_exp2_model1_lora/submission.zip

Experiment identity: Model1 + LoRA(qkv+out+ffn) + BILOU token head
LoRA r=32, alpha=64, dropout=0.1
Base Phase-1 weights frozen; only LoRA matrices + classifier train.
Tokenizer SHA-256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d
Best held-out epoch: 7
Selected conf

## Run order — v3 BEST

1. Attach **Model-1 `enc_full_best (9).pt`** and the verified `tokenizer.json` to Kaggle; select a T4 and run from the top.
2. Before training, verify the outputs for: tokenizer SHA, strict checkpoint load, LoRA module/trainable-parameter count, zero-delta LoRA check, Fold1/Fold2 alignment, prefix-LM compatibility loss, and oracle ceiling.
3. **Actual LoRA fine-tuning begins in Cell 16**, where `🔥 ACTUAL LoRA FINE-TUNING STARTS HERE` is printed.
4. Fold2 selects the best epoch and confidence thresholds using **Strict Macro-F1**.
5. Cell 17 reloads the best adapter, evaluates Fold2, saves `val_prediction.json`, and writes `heldout_result.json`.
6. The final-refit section is **enabled by default** (`DO_FINAL_REFIT_ON_ALL=True`) and recreates Model 1 from the untouched Phase-1 checkpoint, then trains a fresh LoRA adapter on Fold1+Fold2 for exactly `BEST_EPOCH` epochs.
7. Fold3 inference produces `submission_heldout.zip`, `submission_refit.zip`, and the primary `submission.zip` (the final-refit submission).
8. The provided standalone `evaluate_phase2.py` is run against Fold2 gold + held-out predictions as an independent local-score check.

### Experiment identity

This remains **one distinct Phase-2 adaptation experiment: Model1 + LoRA**. The original Model-1 backbone is frozen; only LoRA matrices and the NER classifier are trained. LoRA is injected into all 14 blocks at attention `qkv`, attention `out_proj`, FFN `ff.0`, and FFN `ff.2`.

Submitting both held-out and refit variants can help choose the better hidden-test model, but they are still the **same LoRA experiment**, not two different required techniques.
